# MixesDB search → tracklists

Enter a search word (e.g. `hawtin`). This notebook:
1. Searches MixesDB for mixes whose **title** contains the word (via `mixesdb_scraper.py`)
2. Fetches the tracklist for each matching mix (via `mixesdb_tracklist.py`)
3. Prints the tracklist for every mix that has one, and skips those without
4. Optionally saves all tracklists to a single long table in one XLSX sheet

In [1]:
import sys, os
sys.path.insert(0, os.getcwd())

from mixesdb import search, get_tracklist
import pandas as pd

## 1. Enter the search word

In [2]:
query = 'stussy'  # search query; change to whatever you want
max_mixes = 2000  # how many matching mixes to fetch tracklists for; raise if you want more

print(f"Searching MixesDB for '{query}'...")

Searching MixesDB for 'stussy'...


In [3]:
query

'stussy'

## 2. Run the search (title must contain the word)

In [4]:
results = search(query, limit=max_mixes, strict=True)

print(f"Found {len(results)} mix(es) with '{query}' in the title.\n")
pd.DataFrame(results)[["title", "url"]] if results else print("No matching mixes found.")

Found 20 mix(es) with 'stussy' in the title.



,title,url
0,2024-10-12 - Chris Stussy - Essential Mix,https://www.mixesdb.com/w/2024-10-12_-_Chris_S...
1,"2024-07-11 - Ant TC1 (Stussy), Detboi - Dispat...",https://www.mixesdb.com/w/2024-07-11_-_Ant_TC1...
2,2016-05-31 - Chris Stussy - Boothstock Guestmix 2,https://www.mixesdb.com/w/2016-05-31_-_Chris_S...
3,2016-08-20 - Chris Stussy @ Straf Werk Festiva...,https://www.mixesdb.com/w/2016-08-20_-_Chris_S...
4,2021-01-05 - Chris Stussy - Up The Stuss Radio 02,https://www.mixesdb.com/w/2021-01-05_-_Chris_S...
5,2020-08-20 - Chris Stussy - Awakenings Podcast...,https://www.mixesdb.com/w/2020-08-20_-_Chris_S...
6,2023-06-09 - Chris Stussy & Franky Wah - Globa...,https://www.mixesdb.com/w/2023-06-09_-_Chris_S...
7,2022-04-16 - Chris Stussy @ Awakenings Easter ...,https://www.mixesdb.com/w/2022-04-16_-_Chris_S...
8,2018-08-18 - Prunk & Chris Stussy @ Straf Werk...,https://www.mixesdb.com/w/2018-08-18_-_Prunk_&...
9,"2026-04-16 - Chris Stussy - Residency, BBC Rad...",https://www.mixesdb.com/w/2026-04-16_-_Chris_S...


## 3. Fetch and display the tracklist for each mix

In [5]:
all_tracks = []  # flat rows for the CSV export: mix_title, mix_url, part, position, timestamp, artist, title, raw

for r in results:
    print(f"\n{'='*80}\n{r['title']}\n{r['url']}\n{'='*80}")

    try:
        tracks = get_tracklist(r["url"])
    except Exception as e:
        print(f"  Could not fetch tracklist: {e}")
        continue

    if not tracks:
        print("  No tracklist found on this page.")
        continue

    current_part = None
    for t in tracks:
        if t["part"] != current_part:
            current_part = t["part"]
            if current_part:
                print(f"\n-- {current_part} --")
        ts = f"[{t['timestamp']}] " if t["timestamp"] else ""
        line = f"{t['artist']} - {t['title']}" if t["artist"] else t["raw"]
        print(f"{t['position']:>3}. {ts}{line}")

        all_tracks.append({
            "mix_title": r["title"],
            "mix_url": r["url"],
            **t,
        })

# print(f"\n\nDone. Collected {len(all_tracks)} tracks across {len(results)} mix(es).")


2024-10-12 - Chris Stussy - Essential Mix
https://www.mixesdb.com/w/2024-10-12_-_Chris_Stussy_-_Essential_Mix
  1. [002] Alex Cortex - Discola [Housewax]
  2. [006] Nail - Lemon Gus [Shabby Doll]
  3. [010] Jam & Spoon - Right In The Night (Fall In Love With Music) (MK In The Night Dub) [Epic]
  4. [016] Chris Stussy - The Streets Is Where I'm From
  5. [021] ? - Untitled (B1)
  6. [025] Jex Opolis - Wide Awake (Dub) [Bad Timin']
  7. [030] Paolo Rocco - First Night Out (Chris Stussy Remix)
  8. [035] Burnski & Kepler - Solstice
  9. [039] NairLess - Till It Glows [Radiant]
 10. [044] Underworld - Jumbo (Chris Stussy Edit)
 11. [048] Makam - You Might Lose It (Kerri Chandler Kaoz 6:23 Again Mix) [Sushitech]
 12. [053] Locklead - Apollo
 13. [058] Gaskin - Superstyler
 14. [064] ?
 15. [068] Mandar - String Theory [Oscillat]
 16. [075] Chris Stussy - What Are You Waiting For (Sunrise Mix)
 17. [080] Stussko - Link In The Park [Self Released]
 18. [083] Malin Genie - Phaethon [Up The St

## 4. (optional) Save all tracklists to a single XLSX sheet

In [ ]:
if all_tracks:
    out_path = f"{query.replace(' ', '_')}_tracklists.xlsx"
    df = pd.DataFrame(all_tracks)
    cols = ["mix_title", "mix_url", "part", "position", "timestamp", "artist", "title", "raw"]

    df[cols].to_excel(out_path, sheet_name="tracklists", index=False, engine="openpyxl")

    print(f"Saved {len(df)} rows across {df['mix_title'].nunique()} tracklist(s) to {out_path}")
else:
    print("Nothing to save — no tracklists were found.")